<a href="https://colab.research.google.com/github/RedGummyBear/ImmunomodulatorWerk/blob/main/Benchmark_Matrix_V5toV20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.2/36.2 MB 38.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd, numpy as np, json, math, warnings, io
warnings.filterwarnings('ignore')
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors

In [ ]:
def deltaG(Kd_nM):                     # kcal mol-1, 298 K
    return 1.987e-3 * 298 * math.log(Kd_nM * 1e-9)

def off_target_alert(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return 1.0
    return float(Descriptors.NumAromaticRings(mol) > 2 and Crippen.MolLogP(mol) > 4)

In [ ]:
PUB_IC50 = {                     # nM
    'V5': 3800,                  # MYCMI-6 SPR
    'V9': 500,                   # predicted optimised
    'KJ-Pyr-9': 100,
    'Omomyc': 3,
    '10058-F4': 10000
}

def potency_nM(smiles, alias=None):
    if alias and alias in PUB_IC50:
        return float(PUB_IC50[alias])
    for k, v in PUB_IC50.items():
        if k in smiles:
            return float(v)
    return 500.0                 # fallback dummy

In [ ]:
# ---------- candidate list V5 → V17 ----------
candidates = [
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCCS(=O)(=O)CCO)cc1',           # V5
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)N2CCOCC2)cc1',             # V9
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)Nc1cccnc1)cc1',             # V10
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)N2CCOCC2)c(C(F)(F)F)cc1',   # V11
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)c1ccno1)cc1',               # V12
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)(C)N2CCOCC2)cc1',           # V13
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)n1nc(C)cc1=O)cc1',          # V14
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)(C)n1nc(C)c(C(F)(F)F)c1)cc1',# V15
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCCc1cc2cnoc2n1)cc1',            # V16
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)(C)c1ccc(F)c1-c1cc2cnoc2n1)cc1'  # V17
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)N(c1ccc(OCC(C)O)cc1)c1ccc2ncncc2c1',  # V18
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)(C)c1ccc2ncncc2c1)cc1',    # V19
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)(C)c1ccc(C(F)(F)F)c1-c1cc2ccncc2n1)cc1'  # V20
]
final_names = ['V5', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20']

# ---------- global benchmarks ----------
benchmarks = {
    'Rule-of-5 oral median': 100,
    'First-in-class oncology': 1000,
    'CNS / intracellular': 10,
    'Best-in-class MYC-MAX (Omomyc)': 2,
    'Fragment hit upper': 100000,
    'PAINS discard lower': 10000
}

def evaluate_vs_benchmarks(smiles_list, names):
    rows = []
    for smi, name in zip(smiles_list, names):
        ic50 = potency_nM(smi, alias=name)
        for bench_name, bench_nM in benchmarks.items():
            rows.append({
                'compound': name,
                'IC50_nM': ic50,
                'benchmark': bench_name,
                'target_nM': bench_nM,
                'ΔΔG_kcal': deltaG(ic50) - deltaG(bench_nM),
                'pass': ic50 <= bench_nM
            })
    return pd.DataFrame(rows)

df_full = evaluate_vs_benchmarks(candidates, names)
pivot_full = df_full.pivot(index='compound', columns='benchmark', values='pass')

def colour_pass(val):
    return f'background-color: {"lightgreen" if val else "lightcoral"}'

pivot_full.style.applymap(colour_pass)

benchmark,Best-in-class MYC-MAX (Omomyc),CNS / intracellular,First-in-class oncology,Fragment hit upper,PAINS discard lower,Rule-of-5 oral median
compound,,,,,,
V10,False,False,True,True,True,False
V11,False,False,True,True,True,False
V12,False,False,True,True,True,False
V13,False,False,True,True,True,False
V14,False,False,True,True,True,False
V15,False,False,True,True,True,False
V16,False,False,True,True,True,False
V17,False,False,True,True,True,False
V5,False,False,False,True,True,False


In [ ]:
pivot = df_bm.pivot(index='compound', columns='benchmark', values='pass')

def colour_pass(val):
    return f'background-color: {"lightgreen" if val else "lightcoral"}'

pivot.style.applymap(colour_pass)

benchmark,Best-in-class MYC-MAX (Omomyc),CNS / intracellular,First-in-class oncology,Fragment hit upper,PAINS discard lower,Rule-of-5 oral median
compound,,,,,,
V15,False,False,True,True,True,False
V16,False,False,True,True,True,False
